# CortexCore — PPO Energy Management Agent

Trains a PPO agent for home energy management using the VayuGrid simulator.
The agent controls battery charge/discharge, EV charging, and P2P market bids.

Run with **dry-run** (quick smoke test) or **real data** (Pecan Street profiles).

## 1. Setup

In [ ]:
# ─── Configuration ───
USE_REAL_DATA = False   # set True for Pecan Street real profiles
DATA_CITY = "bangalore"
TOTAL_TIMESTEPS = 200_000 if USE_REAL_DATA else 2048  # 2048 = 1 PPO update for dry-run
CHECKPOINT_DIR = "/content/drive/MyDrive/vayugrid_checkpoints" if USE_REAL_DATA else "checkpoints"

if USE_REAL_DATA:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# ─── Install dependencies ───
!pip install -q gymnasium onnx onnxruntime 'pandas<3' 'numpy<2'

In [ ]:
# ─── Clone repo & install package ───
import os

REPO_DIR = "/content/vayugrid"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/V4RSH1TH-R3DDY/VayuGrid.git {REPO_DIR}
%cd {REPO_DIR}

if USE_REAL_DATA:
    DRIVE_DATA = "/content/drive/MyDrive/vayugrid_data"
    if not os.path.exists("data"):
        !ln -s "{DRIVE_DATA}/data" data

!pip install -e . --no-deps -q
print("Repo ready")

## 2. Training

Runs PPO training loop with proper reward shaping and observation normalization.

In [ ]:
import time
from pathlib import Path

import numpy as np
import torch

from ai.core import (
    CortexCore,
    CortexCoreExporter,
    ObservationNormalizer,
    PPOConfig,
    RewardConfig,
)
from ai.env.gym_env import EnvConfig, VayuGridEnv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

checkpoint_dir = Path(CHECKPOINT_DIR)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# ── Agent ──
normalizer = ObservationNormalizer((12,))
agent = CortexCore(cfg=PPOConfig(), reward=RewardConfig(), normalizer=normalizer, device=device)

# ── Environment ──
env = VayuGridEnv(EnvConfig(
    scenario_path="scenarios/phase1_default.json",
    use_pecan=USE_REAL_DATA,
    city=DATA_CITY,
))
obs, _ = env.reset()

# ── Training loop ──
episode_rewards: list[float] = []
ep_rew = 0.0
ep_len = 0
ep_count = 0
best_mean_reward = -float("inf")
t0 = time.perf_counter()

for step in range(1, TOTAL_TIMESTEPS + 1):
    action_np, log_prob, value = agent.select_action(obs)
    next_obs, env_reward, terminated, truncated, info = env.step(action_np)
    done = terminated or truncated

    # Reward shaping
    soc_norm = obs[0]
    rew, _ = agent.reward_computer.compute(
        p2p_revenue_dollar=info.get("p2p_revenue", 0.0),
        grid_import_cost_dollar=info.get("grid_cost", 0.0),
        soc_norm=soc_norm,
        ev_deadline_missed=False,
        gnn_signal_followed=False,
    )

    agent.push_transition(obs, action_np, rew, value, log_prob, done)
    ep_rew += rew
    ep_len += 1
    obs = next_obs
    normalizer.update(obs)

    if done:
        episode_rewards.append(ep_rew)
        ep_rew = 0.0
        ep_len = 0
        ep_count += 1
        obs, _ = env.reset()

        window = episode_rewards[-20:]
        mean_rew = float(np.mean(window)) if window else 0.0
        if ep_count % 10 == 0:
            elapsed = time.perf_counter() - t0
            sps = step / elapsed
            print(f"  Ep {ep_count:>4d} | step {step:>7d} | mean_reward={mean_rew:+7.2f} | {sps:5.0f} steps/s")

    if agent.buffer.full:
        metrics = agent.update(obs, done)
        elapsed = time.perf_counter() - t0
        sps = step / elapsed
        mean_rew = float(np.mean(episode_rewards[-20:])) if episode_rewards else 0.0
        print(f"  Update at step {step:>7d} | "
              f"policy_loss={metrics['policy_loss']:.4f} "
              f"value_loss={metrics['value_loss']:.4f} "
              f"entropy={metrics['entropy']:.4f} "
              f"mean_reward={mean_rew:+7.2f} | {sps:5.0f} steps/s")

        if mean_rew > best_mean_reward:
            best_mean_reward = mean_rew
            agent.save(str(checkpoint_dir / "cortexcore_best.pt"))

    if step % 50_000 == 0:
        agent.save(str(checkpoint_dir / f"cortexcore_step_{step}.pt"))

# ── Final save ──
final_path = str(checkpoint_dir / "cortexcore_final.pt")
agent.save(final_path)
print(f"\nDone: {TOTAL_TIMESTEPS} steps, {ep_count} episodes")
print(f"Best mean reward: {best_mean_reward:.2f}")

## 3. Export & Download

Export to ONNX and download checkpoints for deployment.

In [ ]:
try:
    exporter = CortexCoreExporter(agent.actor)
    onnx_path = exporter.full_pipeline(
        fp32_path=str(checkpoint_dir / "cortexcore_actor_fp32.onnx"),
        int8_path=str(checkpoint_dir / "cortexcore_actor_int8.onnx"),
    )
    print(f"Exported: {onnx_path}")
except Exception as e:
    print(f"ONNX export skipped: {e}")

In [ ]:
from google.colab import files

for f in ["cortexcore_best.pt", "cortexcore_final.pt", "cortexcore_actor_fp32.onnx", "cortexcore_actor_int8.onnx"]:
    p = checkpoint_dir / f
    if p.exists():
        files.download(str(p))
print("Downloads complete")